In [50]:
from brollm import BaseContract
from broflow import BaseTask, TaskRegistry, Flow
from broskill import SkillControl, ToolControl, Skill, Tool, Arg
from broskill.processing.tool import to_args

from pathlib import Path
import subprocess
import sys
from dataclasses import dataclass
from enum import StrEnum
from typing import Any

ROOT = Path.cwd().resolve().parent 
SKILL_DIR = ROOT / "skills"

sc = SkillControl(SKILL_DIR)
tc = ToolControl(sc)

sc.list_skills()

[Skill(name='read-file', description="Read the contents of a specific file, or list files matching a pattern. Use when the user wants to see what's in a file, or wants to find files matching a pattern.", version='v0.1.0', path=WindowsPath('D:/study-on-agent/skills/read-file'), tags=['filesystem'], keywords=None, default=False, status='experiment'),
 Skill(name='tell-joke', description='Tell a joke on request -- dad jokes, puns, or a mix of both. Use when the user asks for a joke, wants to be entertained, or needs a laugh.', version='v0.1.0', path=WindowsPath('D:/study-on-agent/skills/tell-joke'), tags=['fun', 'entertainment'], keywords=None, default=False, status='experiment')]

In [51]:
tool = tc.load_tool('read-file', 'scripts/read_file.py')
tool

Tool(name='read_file', description="Read exactly one file's content, found via a glob pattern that must match a single file.", args=[Arg(name='pattern', type='string', description="Glob pattern, relative to the project root, that matches exactly one file (e.g. 'skills/tell-joke/references/dad-joke.md').", required=True)], path=WindowsPath('D:/study-on-agent/skills/read-file/scripts/read_file.py'))

In [52]:
def register_tool(tool)->dict:
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": {
                "type": "object",
                "properties": {
                    arg.name: {
                        "type": arg.type,
                        "description": arg.description
                    } for arg in tool.args
                },
                "required": [arg.name for arg in tool.args if arg.required]
            }
        }
    }

In [53]:
class Process(StrEnum):
    INPUT = 'input'
    ROUTER = 'router'
    TOOL_SELECTION = 'tool_selection'
    TOOL_EXECUTION = 'tool_execution'
    ANSWER = 'answer'
    FINISH = 'finish'

@dataclass
class State:
    messages: list
    skills: list
    tools: list

In [54]:
import httpx
from brollm import BaseContract
from typing import Any

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3.8:latest"

def input_fn(
    system_prompt: str,
    messages: list[dict],
    tool_registry: list[dict] | None = None,
) -> dict:
    payload = {
        "model": MODEL,
        "messages": [{"role": "system", "content": system_prompt}, *messages],
        "stream": False,
    }
    if tool_registry:
        payload["tools"] = tool_registry
    resp = httpx.post(OLLAMA_URL, json=payload, timeout=60*4)
    resp.raise_for_status()
    return resp.json()

def output_fn(response: dict) -> Any:
    return response

llm = BaseContract(input_fn=input_fn, output_fn=output_fn)

In [55]:
def UserMessage(content)->dict[str, Any]:
    return {"role":"user", "content":content}

def AssistantMessage(response):
    return response

def ToolResult(tool_name, content):
    return {"role": "tool", "tool_name": tool_name, "content": content}

In [92]:
TOOL_REGISTRYS = [
    {
        "type": "function",
        "function": {
            "name": "load_skill",
            "description": (
                "Load the full instructions for a registered skill by name. "
                "Call this only when the current task clearly matches that "
                "skill's description."
            ),
            "parameters": {
                "type": "object",
                "properties": {"skill_name": {"type": "string"}},
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "load_skill_extension",
            "description": (
                "Load the full contents of one reference/asset document belonging to "
                "an already-loaded skill. Call this only when that skill's "
                "instructions point you to a specific reference/asset file for more "
                "detail — don't call it speculatively."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "skill_name": {
                        "type": "string",
                        "description": "The name of the skill whose reference/asset you want to load."
                    },
                    "path": {
                        "type": "string",
                        "description": "The reference/asset file's path exactly as shown in the skill's instructions, e.g. 'references/aws.md' or `assets/color.ts`."
                    }
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "ask_followup_question",
            "description": (
                "Signal that you need the user to answer something before you can "
                "continue. Write the actual question as your normal response "
                "content, then call this tool with no arguments to pause the turn "
                "and wait for their reply."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },        
]

In [93]:
TOOLS = {
    "load_skill_extension": sc.load_skill_extension,
    "load_skill": sc.load_skill,
    "ask_followup_question": None
}

In [85]:
# # tool = tc.load_tool('read-file', 'scripts/read_file.py')
# TOOLS = [register_tool(tc.load_tool('read-file', tool)) for tool in ['scripts/list_files.py', 'scripts/read_file.py']]
# TOOLS

In [101]:
system_prompt = "you're a helpful assistant"
# content = "Read broai/main.py for me."
# content = "read everything in skills/*.md"
content = "I'm so bored. Can you tell me a joke?"
messages = [UserMessage(content)]
available_skills = "\n".join([f"- {s.name}: {s.description}" for s in sc.list_skills()])
messages.append(ToolResult('list_skills', available_skills))
# skill_content = sc.load_skill_extension("tell-joke", "SKILL.md")
# messages.append(ToolResult(tool_name='load_skill', content=skill_content))
# response = llm(system_prompt, messages, tool_registry=TOOL_REGISTRYS)
# response

In [102]:
while True:
    response = llm(system_prompt, messages, tool_registry=TOOL_REGISTRYS)
    messages.append(response['message'])
    message = response['message']
    _content = message.get('content', '')
    tool_calls = message.get('tool_calls', [])
    if not tool_calls:
        break
    for t in tool_calls:
        tool_name = t['function']['name']
        tool_args = t['function']['arguments']
        if tool_name=='ask_followup_question':
            if not _content:
                messages.append(ToolResult('ask_followup_question', 'Do not leave message blank. Ask users for more clarification'))
                continue
            else:
                clarified_msg = input(_content)
                messages.append(ToolResult('ask_followup_question', clarified_msg))
                continue
        if tool_name in TOOLS:
            result = TOOLS[tool_name](**tool_args)
            messages.append(ToolResult(tool_name, str(result)))
        else:
            print(f"Tool {tool_name} not found in registry.")
    # break

KeyboardInterrupt: 

In [104]:
messages

[{'role': 'user', 'content': "I'm so bored. Can you tell me a joke?"},
 {'role': 'tool',
  'tool_name': 'list_skills',
  'content': "- read-file: Read the contents of a specific file, or list files matching a pattern. Use when the user wants to see what's in a file, or wants to find files matching a pattern.\n- tell-joke: Tell a joke on request -- dad jokes, puns, or a mix of both. Use when the user asks for a joke, wants to be entertained, or needs a laugh."},
 {'role': 'assistant',
  'content': '',
  'thinking': 'The user is asking for a joke. There\'s a "tell-joke" skill registered that matches this request. Let me load it to get the proper instructions for telling a joke.',
  'tool_calls': [{'id': 'call_e76eb183',
    'function': {'index': 0,
     'name': 'load_skill',
     'arguments': {'skill_name': 'tell-joke'}}}]},
 {'role': 'tool',
  'tool_name': 'load_skill',
  'content': "# Tell Joke\n\n## Instructions\n\n- Ask the user which kind of joke they'd like: dad jokes, puns, or a m

In [87]:
messages.append(response['message'])

In [88]:
messages.append(ToolResult('ask_followup_question', 'I think dad joke about Cowbow would be fun.'))
response = llm(system_prompt, messages, tool_registry=TOOL_REGISTRYS)

In [89]:
response

{'model': 'qwen3.8:latest',
 'created_at': '2026-09-13T17:19:29.0144999Z',
 'message': {'role': 'assistant',
  'content': '',
  'thinking': 'The user wants a dad joke about a cowboy. Let me load the dad joke reference file to find a good one.',
  'tool_calls': [{'id': 'call_frzyv0xy',
    'function': {'index': 0,
     'name': 'load_skill_extension',
     'arguments': {'skill_name': 'tell-joke',
      'path': 'references/dad-joke.md'}}}]},
 'done': True,
 'done_reason': 'stop',
 'total_duration': 3739952800,
 'load_duration': 2947700,
 'prompt_eval_count': 924,
 'prompt_eval_cached_count': 899,
 'prompt_eval_duration': 805622000,
 'eval_count': 74,
 'eval_duration': 2924887000}

In [81]:
messages.append(ToolResult('load_skill_extension', sc.load_skill_extension('tell-joke', 'references/dad-joke.md')))

In [82]:
response = llm(system_prompt, messages, tool_registry=TOOL_REGISTRYS)
messages.append(response['message'])

In [83]:
response

{'model': 'qwen3.8:latest',
 'created_at': '2026-09-13T17:17:32.1459397Z',
 'message': {'role': 'assistant',
  'content': "Why don't cowboys make good tennis players?\n\nBecause they always draw their gun before serving. 🤠",
  'thinking': "The user wants a dad joke about a cowboy. Looking at the reference, there's no cowboy-specific one listed, so I need to come up with one in my own words, keeping it short and not explaining it."},
 'done': True,
 'done_reason': 'stop',
 'total_duration': 5161654000,
 'load_duration': 3435500,
 'prompt_eval_count': 1030,
 'prompt_eval_cached_count': 856,
 'prompt_eval_duration': 823152000,
 'eval_count': 71,
 'eval_duration': 4325851000}